# Train Vision Transformer (ViT) on miniature dataset

This notebook trains the `SpectralViT` model on the miniature noiseless dataset
generated by `SimpleSimulator`. The ViT treats each spatial patch of the
spectrogram as a token and uses multi-head self-attention to capture long-range
cross-dispersion correlations before decoding to a 3-D spectral cube.

## Architecture recap
- **Input**: 4 (or 5) x H_spec x W_spec spectrogram tensor.
- **Patch embedding**: splits the 2-D input into non-overlapping patches
  (default: 8x8 pixels), linearly embeds each patch into a token.
- **Transformer encoder**: `depth=6` blocks of multi-head self-attention + MLP.
- **Decoder head**: each token is projected to `patch_size² x n_lambda` values
  then pixel-shuffled back to the full spatial resolution.
- **Output**: `(n_lambda, ny, nx)` spectral cube, center-cropped from the
  padded spectrogram dimensions.

In [ ]:
# ── Imports ────────────────────────────────────────────────────────────────
from pathlib import Path
import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt

import spectangle
from spectangle.data_loaders.dataset import SpectangleDataset, SpectangleDataModule
from spectangle.models.vit import SpectralViT
from spectangle.models.losses import CombinedLoss
from spectangle.utils.metrics import cube_metrics
from spectangle.utils.training import get_device

# ── Config ─────────────────────────────────────────────────────────────────
# Resolve H5 path relative to the project root so the notebook works
# regardless of the current working directory it is launched from.

PATCH_SIZE    = 8
EMBED_DIM     = 128      # reduced for fast notebook runs
DEPTH         = 4
N_HEADS       = 4
N_LAMBDA      = 128
IN_CHANNELS   = 5        # use direct image as spatial prior
LR            = 5e-5
N_EPOCHS      = 5        # increase for real training
BATCH_SIZE    = 4
DEVICE        = get_device()

print(f'H5 path  : {H5_PATH}')
print(f'Exists   : {H5_PATH.exists()}')
print(f'Device   : {DEVICE}')


## 1. Load data and inspect geometry

In [ ]:
# ── Imports ────────────────────────────────────────────────────────────────
from pathlib import Path
import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt

import spectangle
from spectangle.data_loaders.dataset import SpectangleDataset, SpectangleDataModule
from spectangle.models.vit import SpectralViT
from spectangle.models.losses import CombinedLoss
from spectangle.utils.metrics import cube_metrics
from spectangle.utils.training import get_device

# ── Config ─────────────────────────────────────────────────────────────────
# Resolve H5 path relative to the project root so the notebook works
# regardless of the current working directory it is launched from.

PATCH_SIZE    = 8
EMBED_DIM     = 128      # reduced for fast notebook runs
DEPTH         = 4
N_HEADS       = 4
N_LAMBDA      = 128
IN_CHANNELS   = 5        # use direct image as spatial prior
LR            = 5e-5
N_EPOCHS      = 5        # increase for real training
BATCH_SIZE    = 4
DEVICE        = get_device()

print(f'H5 path  : {H5_PATH}')
print(f'Exists   : {H5_PATH.exists()}')
print(f'Device   : {DEVICE}')


## 2. Build the ViT model

In [ ]:
model = SpectralViT(
    in_channels=IN_CHANNELS,
    image_size_h=H_spec,
    image_size_w=W_spec,
    patch_size=PATCH_SIZE,
    embed_dim=EMBED_DIM,
    depth=DEPTH,
    n_heads=N_HEADS,
    n_lambda=N_LAMBDA,
    scene_shape=scene_shape,
).to(DEVICE)

print(f'Parameters: {model.parameter_count():,}')

# Quick sanity check: forward pass
x_dummy = torch.zeros(2, IN_CHANNELS, H_spec, W_spec, device=DEVICE)
y_dummy = model(x_dummy)
print(f'Output shape: {y_dummy.shape}  (expected: [2, {N_LAMBDA}, {scene_shape[0]}, {scene_shape[1]}])')

## 3. Training loop

In [ ]:
loss_fn   = CombinedLoss(weights={'mse': 1.0, 'ssim': 0.5, 'spectral': 0.1})
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=N_EPOCHS)

train_loader = dm.train_dataloader()
val_loader   = dm.val_dataloader()

def run_epoch(loader, train=True):
    model.train() if train else model.eval()
    total = 0.0
    n = 0
    with torch.set_grad_enabled(train):
        for x, y in loader:
            # Pad spectrograms to ViT grid if needed
            x_in = x.to(DEVICE)
            pad_h = H_spec - x_in.shape[-2]
            pad_w = W_spec - x_in.shape[-1]
            if pad_h > 0 or pad_w > 0:
                x_in = F.pad(x_in, (0, pad_w, 0, pad_h))
            y    = y.to(DEVICE)
            pred = model(x_in)
            loss, _ = loss_fn(pred, y)
            if train:
                optimizer.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
            total += loss.item() * x.shape[0]
            n     += x.shape[0]
    return total / max(n, 1)

train_losses, val_losses = [], []
for epoch in range(1, N_EPOCHS + 1):
    tl = run_epoch(train_loader, train=True)
    vl = run_epoch(val_loader,   train=False)
    scheduler.step()
    train_losses.append(tl)
    val_losses.append(vl)
    lr_now = optimizer.param_groups[0]['lr']
    print(f'Epoch {epoch:3d}/{N_EPOCHS} | train={tl:.5f} | val={vl:.5f} | lr={lr_now:.2e}')

# Plot loss curves
fig, ax = plt.subplots(figsize=(6, 3))
ax.plot(train_losses, label='train')
ax.plot(val_losses,   label='val')
ax.set_xlabel('Epoch'); ax.set_ylabel('Loss'); ax.legend(); ax.grid(True)
ax.set_title('ViT training curves (miniature)')
plt.tight_layout(); plt.show()

## 4. Evaluate a reconstruction
Predict a cube for the first validation sample and compare it to the
ground truth visually and with quantitative metrics (PSNR, SSIM, SAM, RMSE).

In [ ]:
from spectangle.utils.metrics import cube_metrics

model.eval()
x_s, y_s = dm._val_ds[0]
x_in = x_s.unsqueeze(0).to(DEVICE)

# Pad to ViT grid
pad_h = H_spec - x_in.shape[-2]
pad_w = W_spec - x_in.shape[-1]
if pad_h > 0 or pad_w > 0:
    x_in = F.pad(x_in, (0, pad_w, 0, pad_h))

with torch.no_grad():
    pred = model(x_in).squeeze(0).cpu().numpy()  # (n_lambda, ny, nx)

gt = y_s.numpy()  # (n_lambda, ny, nx)
metrics = cube_metrics(pred, gt)
print('Reconstruction metrics:')
for k, v in metrics.items():
    print(f'  {k}: {v:.4f}')

# Visual comparison — broadband
fig, axs = plt.subplots(1, 3, figsize=(12, 4))
axs[0].imshow(gt.sum(0),            cmap='gray',  origin='lower'); axs[0].set_title('GT broadband')
axs[1].imshow(pred.sum(0),          cmap='gray',  origin='lower'); axs[1].set_title('Pred broadband')
axs[2].imshow(np.abs(pred.sum(0) - gt.sum(0)), cmap='magma', origin='lower'); axs[2].set_title('|Residual|')
for a in axs: a.axis('off')
plt.tight_layout(); plt.show()

# Spectral comparison at brightest pixel
iy, ix = np.unravel_index(np.argmax(gt.sum(0)), gt.shape[1:])
wav = np.linspace(9250, 18500, N_LAMBDA)
fig, ax = plt.subplots(figsize=(7, 3))
ax.plot(wav, gt[:, iy, ix],   label='Ground truth', lw=1.5)
ax.plot(wav, pred[:, iy, ix], label='ViT prediction', lw=1.2, ls='--')
ax.set_xlabel('Wavelength (Å)'); ax.set_ylabel('Flux (a.u.)'); ax.legend(); ax.grid(True)
ax.set_title(f'Spectrum at brightest pixel ({ix},{iy})')
plt.tight_layout(); plt.show()

## 5. Attention map visualisation (optional)
The ViT's self-attention heads can tell us which spatial patches are most
influential for the reconstruction. This is a powerful debugging tool: if
attention heads focus on noise or padding regions, the model may be overfitting
to artefacts rather than physical signal.

> **Note**: Extracting attention maps requires a small hook on the
> `TransformerBlock.attn` module. The code below shows how to do this.

In [ ]:
# Hook to capture attention weights from the last Transformer block
attn_weights = {}
def _save_attn(module, inp, out):
    # MultiheadAttention returns (output, attn_weights)
    attn_weights['last'] = out[1].detach().cpu()

# Register hook on the LAST transformer block's attention
hook = model.transformer[-1].attn.register_forward_hook(_save_attn)

model.eval()
with torch.no_grad():
    _ = model(x_in)

hook.remove()

if 'last' in attn_weights:
    # Mean over heads and take the attention from the first token (CLS-like)
    attn = attn_weights['last']  # (B, n_heads, N, N)
    attn_mean = attn[0].mean(0)[0].numpy()  # (N,)
    n_ph = model.n_patches_h
    n_pw = model.n_patches_w
    attn_map = attn_mean.reshape(n_ph, n_pw)

    fig, ax = plt.subplots(figsize=(5, 5))
    ax.imshow(attn_map, cmap='hot', origin='lower')
    ax.set_title('Mean attention (last layer, from first token)')
    ax.set_xlabel('Patch column'); ax.set_ylabel('Patch row')
    plt.tight_layout(); plt.show()
else:
    print('Attention weights not captured (check hook).')

## Next steps
- Run with `N_EPOCHS=150` and `EMBED_DIM=256` via the script:
  ```
  python scripts/train.py --config configs/models/vit.yaml
  ```
- Compare ViT vs U-Net metrics at equal parameter budgets.
- Enable 5-channel input (`IN_CHANNELS=5`) to leverage the direct image
  as a spatial prior — often boosts reconstruction quality at low SNR.